# 中观景气视角行业轮动策略

基于华泰证券金工深度研究报告《行业配置策略：中观景气视角》的量化策略复现

**核心方法**：
1. 行业指标库构建
2. 指标预处理（总量类、价格类、同比类）
3. Simple-Nowcasting模型生成景气指数
4. 行业轮动策略

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print('Libraries imported successfully')

## 1. 配置和数据初始化

In [ ]:
from source.config import (
    TUSHARE_TOKEN, TUSHARE_API_URL,
    CITIC_INDUSTRIES, CITIC_CODES,
    ROLLING_WINDOW, TOP_K_INDICATORS, SELECTED_K_INDICATORS,
    STRATEGY_TOP_N, BACKTEST_START, BACKTEST_END
)

print('配置参数:')
print(f'  回测区间: {BACKTEST_START} - {BACKTEST_END}')
print(f'  滚动窗口: {ROLLING_WINDOW}个月')
print(f'  代理指标数量: {TOP_K_INDICATORS}')
print(f'  策略持仓: Top {STRATEGY_TOP_N}行业')

In [ ]:
from source.data_fetcher import DataFetcher

fetcher = DataFetcher()
print('DataFetcher initialized successfully')

In [ ]:
# 获取月末交易日
monthly_dates = fetcher.get_month_end_dates('2016-04-01', '2022-06-30')
print(f'月末交易日数量: {len(monthly_dates)}')
print(f'首个月末: {monthly_dates[0]}')
print(f'末个月末: {monthly_dates[-1]}')

## 2. 行业指标库

In [ ]:
from source.indicator_lib import IndicatorLibrary, get_citici_code_mapping

indicator_lib = IndicatorLibrary()
citic_codes = get_citici_code_mapping()

print('中信行业代码映射:')
for industry, code in citic_codes.items():
    print(f'  {industry}: {code}')

In [ ]:
# 查看各行业的指标数量
print('\n各行业指标库规模:')
for industry in CITIC_INDUSTRIES[:8]:
    count = indicator_lib.get_indicator_count(industry)
    print(f'  {industry}: {count}个指标')

## 3. 指标预处理示例

In [ ]:
from source.preprocessing import Preprocessor, IndicatorPreprocessor

preprocessor = Preprocessor(rolling_window=ROLLING_WINDOW)
validator = IndicatorPreprocessor(rolling_window=ROLLING_WINDOW, min_valid_length=36)

# 生成模拟指标数据演示
np.random.seed(42)
n = 80
dates = pd.date_range('2015-01-01', periods=n, freq='M')

# 模拟总量类指标
total_indicator = pd.Series(
    np.random.randn(n).cumsum() * 10 + 100,
    index=dates,
    name='total_indicator'
)

# 模拟价格类指标
price_indicator = pd.Series(
    np.random.randn(n).cumsum() * 5 + 50,
    index=dates,
    name='price_indicator'
)

# 预处理
processed_total = preprocessor.preprocess_total_indicator(total_indicator)
processed_price = preprocessor.preprocess_price_indicator(price_indicator)

print('指标预处理示例:')
print(f'原始总量指标前5个值: {total_indicator.head().values}')
print(f'预处理后总量指标前5个值: {processed_total.head().values}')

In [ ]:
# 可视化预处理结果
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 原始总量指标
axes[0, 0].plot(total_indicator.index, total_indicator.values)
axes[0, 0].set_title('Raw Total Indicator')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Value')

# 预处理后总量指标
axes[0, 1].plot(processed_total.index, processed_total.values, color='orange')
axes[0, 1].set_title('Preprocessed Total Indicator (YoY)')
axes[0, 1].set_xlabel('Date')
axes[0, 1].set_ylabel('YoY Growth (%)')

# 原始价格指标
axes[1, 0].plot(price_indicator.index, price_indicator.values, color='green')
axes[1, 0].set_title('Raw Price Indicator')
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('Value')

# 预处理后价格指标
axes[1, 1].plot(processed_price.index, processed_price.values, color='red')
axes[1, 1].set_title('Preprocessed Price Indicator (YoY)')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('YoY Growth (%)')

plt.tight_layout()
plt.savefig('../output/indicator_preprocessing.png', dpi=150)
plt.show()

print('图表已保存到 output/indicator_preprocessing.png')

## 4. Simple-Nowcasting模型

In [ ]:
from source.nowcasting import (
    SimpleNowcasting, IndicatorEvaluator, 
    IndicatorSelector, NowcastingModel
)

# 模拟财务参照数据
np.random.seed(123)
dates = pd.date_range('2016-01-01', periods=78, freq='M')

# 模拟ROE_TTM同比
roe_yoy = pd.Series(
    np.random.randn(78).cumsum() * 2 + 5,
    index=dates,
    name='ROE_TTM_yoy'
)

# 模拟代理指标
n_indicators = 10
indicators = {}
for i in range(n_indicators):
    indicators[f'indicator_{i}'] = pd.Series(
        roe_yoy.values + np.random.randn(78) * 1.5,
        index=dates,
        name=f'indicator_{i}'
    )

print(f'财务参照序列长度: {len(roe_yoy)}')
print(f'代理指标数量: {n_indicators}')

In [ ]:
# 创建Nowcasting模型
model = NowcastingModel(
    rolling_window=ROLLING_WINDOW,
    min_valid_length=36,
    top_k_indicators=SELECTED_K_INDICATORS
)

# 拟合模型
prosperity_index, selected_indicators, scores = model.fit(indicators, roe_yoy)

print(f'选取的代理指标数量: {len(selected_indicators)}')
print(f'选取的指标: {selected_indicators}')
print(f'\n景气指数前10个值:\n{prosperity_index.head(10)}')

In [ ]:
# 计算相关系数
correlation = prosperity_index.corr(roe_yoy)
print(f'景气指数与财务参照的相关系数: {correlation:.4f}')

In [ ]:
# 可视化景气指数与财务参照
fig, ax = plt.subplots(figsize=(12, 5))

ax2 = ax.twinx()

ax.plot(prosperity_index.index, prosperity_index.values, 
        label='Prosperity Index', color='blue', linewidth=2)
ax2.plot(roe_yoy.index, roe_yoy.values, 
         label='ROE YoY (Reference)', color='red', linewidth=2, alpha=0.7)

ax.set_xlabel('Date')
ax.set_ylabel('Prosperity Index', color='blue')
ax2.set_ylabel('ROE YoY (%)', color='red')
ax.set_title(f'Prosperity Index vs ROE YoY (Correlation: {correlation:.3f})')

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.savefig('../output/prosperity_index.png', dpi=150)
plt.show()

print('图表已保存到 output/prosperity_index.png')

## 5. 行业轮动策略

In [ ]:
from source.strategy import IndustryRotationStrategy, MultiFactorStrategy

# 模拟多行业景气指数
industries = CITIC_INDUSTRIES[:6]
dates = pd.date_range('2019-01-01', periods=42, freq='M')

prosperity_dict = {}
for industry in industries:
    base = np.random.rand() * 10
    prosperity_dict[industry] = pd.Series(
        np.random.randn(42).cumsum() * 1.5 + base,
        index=dates,
        name=industry
    )

print(f'行业数量: {len(prosperity_dict)}')
print(f'行业列表: {list(prosperity_dict.keys())}')

In [ ]:
# 计算景气得分
strategy = IndustryRotationStrategy(top_n=STRATEGY_TOP_N)
scores_df = strategy.calculate_prosperity_score(prosperity_dict)

print('行业景气得分排名:')
print(scores_df.sort_values('total_score', ascending=False).to_string())

In [ ]:
# 选取Top行业
selected = strategy.select_industries(scores_df, top_n=STRATEGY_TOP_N)
print(f'\n选取的行业: {selected}')

## 6. 回测演示

In [ ]:
from source.backtest import Backtester, PerformanceAnalyzer
from source.backtest import calculate_equal_weight_returns, generate_performance_report

# 模拟行业收益率
dates = pd.date_range('2019-01-01', periods=42, freq='M')
n = len(dates)

industry_returns = {}
for industry in industries:
    industry_returns[industry] = pd.Series(
        np.random.randn(n) * 0.05 + 0.01,
        index=dates
    )

# 等权组合收益
equal_weight_returns = calculate_equal_weight_returns(industry_returns)

print(f'模拟行业收益率数据形状: {len(equal_weight_returns)}个月')

In [ ]:
# 创建回测器
backtester = Backtester(initial_capital=1000000, transaction_cost=0.001)
analyzer = PerformanceAnalyzer(risk_free_rate=0.03)

# 计算业绩指标
metrics = analyzer.calculate_metrics(equal_weight_returns)

print('\n等权组合业绩指标:')
print(f'  年化收益率: {metrics.annual_return:.2%}')
print(f'  年化波动率: {metrics.annual_volatility:.2%}')
print(f'  夏普比率: {metrics.sharpe_ratio:.2f}')
print(f'  最大回撤: {metrics.max_drawdown:.2%}')
print(f'  卡玛比率: {metrics.calmar_ratio:.2f}')
print(f'  胜率: {metrics.win_rate:.2%}')

In [ ]:
# 计算累计收益
cumulative_returns = analyzer.calculate_cumulative_returns(equal_weight_returns)
drawdown = analyzer.calculate_drawdown(cumulative_returns)

# 可视化
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# 累计收益
axes[0].plot(cumulative_returns.index, cumulative_returns.values, 
             label='Portfolio', color='blue', linewidth=2)
axes[0].set_title('Cumulative Returns')
axes[0].set_ylabel('Cumulative Return')
axes[0].legend()
axes[0].grid(True)

# 回撤
axes[1].fill_between(drawdown.index, drawdown.values * 100, 0, 
                    alpha=0.3, color='red')
axes[1].set_title('Drawdown')
axes[1].set_ylabel('Drawdown (%)')
axes[1].grid(True)

# 月度收益分布
axes[2].bar(equal_weight_returns.index, equal_weight_returns.values * 100)
axes[2].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[2].set_title('Monthly Returns')
axes[2].set_ylabel('Return (%)')
axes[2].grid(True)

plt.tight_layout()
plt.savefig('../output/backtest_results.png', dpi=150)
plt.show()

print('图表已保存到 output/backtest_results.png')

## 7. 多行业轮动策略回测

In [ ]:
# 生成更完整的模拟数据
np.random.seed(42)
dates = pd.date_range('2016-04-01', '2022-06-30', freq='M')
n = len(dates)

# 所有行业
all_industries = CITIC_INDUSTRIES

# 模拟行业收益率（带有行业轮动特征）
industry_returns = {}
for i, industry in enumerate(all_industries):
    # 每个行业有不同的基准收益和波动
    base_return = 0.008 + 0.002 * np.sin(i * 0.5)
    volatility = 0.04 + 0.02 * np.cos(i * 0.3)
    
    returns = pd.Series(
        np.random.randn(n) * volatility + base_return,
        index=dates
    )
    industry_returns[industry] = returns

print(f'行业数量: {len(industry_returns)}')
print(f'时间跨度: {dates[0]} 至 {dates[-1]}')
print(f'数据点数: {n}')

In [ ]:
# 模拟各行业的景气指数
prosperity_dict = {}
for i, industry in enumerate(all_industries):
    # 景气指数带有趋势和周期性
    trend = np.linspace(0, 5, n) + np.sin(i * 0.8) * 3
    cycle = np.sin(np.linspace(0, 4 * np.pi, n)) * 2
    noise = np.random.randn(n) * 0.5
    
    prosperity = pd.Series(
        trend + cycle + noise,
        index=dates,
        name=industry
    )
    prosperity_dict[industry] = prosperity

print('行业景气指数模拟完成')

In [ ]:
# 生成每月调仓信号
strategy = IndustryRotationStrategy(top_n=STRATEGY_TOP_N)

rebalance_dates = []
selected_history = []

for i in range(12, n):
    current_date = dates[i]
    
    # 使用过去12个月的景气指数
    window_prosperity = {
        ind: prof.iloc[:i+1] for ind, prof in prosperity_dict.items()
    }
    
    # 计算得分并选取
    scores = strategy.calculate_prosperity_score(window_prosperity)
    selected = strategy.select_industries(scores, top_n=STRATEGY_TOP_N)
    
    rebalance_dates.append(current_date)
    selected_history.append(selected)

print(f'调仓次数: {len(selected_history)}')
print(f'策略持仓: 每月选取Top {STRATEGY_TOP_N}行业')

In [ ]:
# 计算策略收益
strategy_returns = []

for i in range(len(rebalance_dates)):
    current_date = rebalance_dates[i]
    selected = selected_history[i]
    
    # 分配权重
    weights = [1.0 / len(selected)] * len(selected)
    
    # 计算下期收益
    if i < len(rebalance_dates) - 1:
        next_date = rebalance_dates[i + 1]
    else:
        next_date = dates[-1] + pd.DateOffset(months=1)
    
    # 获取该区间收益
    period_mask = (equal_weight_returns.index > current_date) & \
                  (equal_weight_returns.index <= next_date)
    
    if period_mask.sum() > 0:
        period_returns = equal_weight_returns[period_mask]
        
        # 组合收益
        for idx in period_returns.index:
            daily_ret = 0
            for ind, w in zip(selected, weights):
                if ind in industry_returns:
                    daily_ret += industry_returns[ind].loc[idx] * w
            strategy_returns.append({
                'date': idx,
                'return': daily_ret
            })

strategy_returns_df = pd.DataFrame(strategy_returns).set_index('date')
strategy_returns_series = strategy_returns_df['return']

print(f'策略收益序列长度: {len(strategy_returns_series)}个月')

In [ ]:
# 计算benchmark收益（等权组合）
benchmark_returns = calculate_equal_weight_returns(industry_returns)

# 使用与策略相同的时间段
common_dates = strategy_returns_series.index.intersection(benchmark_returns.index)
strategy_common = strategy_returns_series.loc[common_dates]
benchmark_common = benchmark_returns.loc[common_dates]

print(f'共同时间区间: {common_dates[0]} 至 {common_dates[-1]}')

In [ ]:
# 计算业绩指标
strategy_analyzer = PerformanceAnalyzer(risk_free_rate=0.03)
benchmark_analyzer = PerformanceAnalyzer(risk_free_rate=0.03)

strategy_metrics = strategy_analyzer.calculate_metrics(strategy_common)
benchmark_metrics = benchmark_analyzer.calculate_metrics(benchmark_common)

print('\n策略业绩 vs 基准业绩:')
print(f'{"指标":<20} {"策略":>15} {"基准":>15} {"超额":>15}')
print('-' * 65)
print(f'{"年化收益率":<20} {strategy_metrics.annual_return:>14.2%} {benchmark_metrics.annual_return:>14.2%} {(strategy_metrics.annual_return-benchmark_metrics.annual_return):>14.2%}')
print(f'{"年化波动率":<20} {strategy_metrics.annual_volatility:>14.2%} {benchmark_metrics.annual_volatility:>14.2%}')
print(f'{"夏普比率":<20} {strategy_metrics.sharpe_ratio:>15.2f} {benchmark_metrics.sharpe_ratio:>15.2f} {(strategy_metrics.sharpe_ratio-benchmark_metrics.sharpe_ratio):>15.2f}')
print(f'{"最大回撤":<20} {strategy_metrics.max_drawdown:>14.2%} {benchmark_metrics.max_drawdown:>14.2%}')
print(f'{"卡玛比率":<20} {strategy_metrics.calmar_ratio:>15.2f} {benchmark_metrics.calmar_ratio:>15.2f}')
print(f'{"胜率":<20} {strategy_metrics.win_rate:>15.2%} {benchmark_metrics.win_rate:>15.2%}')

In [ ]:
# 可视化比较
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# 累计收益对比
strategy_cum = analyzer.calculate_cumulative_returns(strategy_common)
benchmark_cum = analyzer.calculate_cumulative_returns(benchmark_common)

axes[0].plot(strategy_cum.index, strategy_cum.values, 
             label='Meso Prosperity Strategy', color='blue', linewidth=2)
axes[0].plot(benchmark_cum.index, benchmark_cum.values, 
             label='Equal Weight Benchmark', color='gray', linewidth=2, alpha=0.7)
axes[0].set_title('Cumulative Returns: Strategy vs Benchmark')
axes[0].set_ylabel('Cumulative Return')
axes[0].legend()
axes[0].grid(True)

# 超额收益
excess_returns = strategy_common - benchmark_common
excess_cum = (1 + excess_returns).cumprod()

axes[1].fill_between(excess_cum.index, excess_cum.values, 1, 
                    where=excess_cum.values >= 1, 
                    alpha=0.3, color='green', label='Outperformance')
axes[1].fill_between(excess_cum.index, excess_cum.values, 1, 
                    where=excess_cum.values < 1, 
                    alpha=0.3, color='red', label='Underperformance')
axes[1].plot(excess_cum.index, excess_cum.values, color='blue', linewidth=1.5)
axes[1].axhline(y=1, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Excess Returns (Strategy - Benchmark)')
axes[1].set_ylabel('Excess Return')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('../output/strategy_comparison.png', dpi=150)
plt.show()

print('图表已保存到 output/strategy_comparison.png')

## 8. 分年度业绩统计

In [ ]:
# 分年度统计
def calculate_annual_stats(returns):
    returns['year'] = returns.index.year
    annual_stats = returns.groupby('year').apply(
        lambda x: pd.Series({
            'annual_return': (1 + x['return']).prod() - 1,
            'volatility': x['return'].std() * np.sqrt(12),
            'sharpe': (x['return'].mean() - 0.03/12) / x['return'].std() * np.sqrt(12) if x['return'].std() > 0 else 0,
            'max_dd': ((1 + x['return']).cumprod() - (1 + x['return']).cumprod().cummax()).min()
        })
    )
    return annual_stats

strategy_annual = calculate_annual_stats(pd.DataFrame({'return': strategy_common}))
benchmark_annual = calculate_annual_stats(pd.DataFrame({'return': benchmark_common}))

print('\n分年度业绩对比:')
print(f'{"年份":<8} {"策略收益":>12} {"基准收益":>12} {"超额收益":>12}')
print('-' * 48)
for year in strategy_annual.index:
    s_ret = strategy_annual.loc[year, 'annual_return']
    b_ret = benchmark_annual.loc[year, 'annual_return']
    excess = s_ret - b_ret
    print(f'{year:<8} {s_ret:>11.2%} {b_ret:>11.2%} {excess:>11.2%}')

## 9. 持仓分析

In [ ]:
# 统计各行业被选中的次数
from collections import Counter

all_selected = [ind for selected in selected_history for ind in selected]
industry_counts = Counter(all_selected)

# 排序
sorted_counts = sorted(industry_counts.items(), key=lambda x: x[1], reverse=True)

print('行业被选中次数排名 (Top 10):')
for industry, count in sorted_counts[:10]:
    print(f'  {industry}: {count}次')

In [ ]:
# 可视化行业选择频率
fig, ax = plt.subplots(figsize=(12, 6))

industries_top = [x[0] for x in sorted_counts[:10]]
counts_top = [x[1] for x in sorted_counts[:10]]

bars = ax.barh(industries_top, counts_top, color='steelblue')
ax.set_xlabel('Selection Count')
ax.set_title('Industry Selection Frequency (Top 10)')
ax.invert_yaxis()

for bar, count in zip(bars, counts_top):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, 
            str(count), va='center')

plt.tight_layout()
plt.savefig('../output/industry_selection.png', dpi=150)
plt.show()

print('图表已保存到 output/industry_selection.png')

## 10. 总结

In [ ]:
print('\n' + '=' * 70)
print('中观景气视角行业轮动策略 - 复现结果总结')
print('=' * 70)
print()
print('策略概述:')
print(f'  - 核心方法: Simple-Nowcasting模型生成中观行业景气指数')
print(f'  - 行业覆盖: {len(all_industries)}个中信行业')
print(f'  - 策略持仓: 每月选取Top {STRATEGY_TOP_N}行业等权配置')
print(f'  - 回测区间: {common_dates[0].strftime("%Y-%m")} 至 {common_dates[-1].strftime("%Y-%m")}')
print()
print('业绩表现:')
print(f'  - 年化收益率: {strategy_metrics.annual_return:.2%}')
print(f'  - 年化波动率: {strategy_metrics.annual_volatility:.2%}')
print(f'  - 夏普比率: {strategy_metrics.sharpe_ratio:.2f}')
print(f'  - 最大回撤: {strategy_metrics.max_drawdown:.2%}')
print(f'  - 卡玛比率: {strategy_metrics.calmar_ratio:.2f}')
print()
print('相对于基准 (等权组合):')
print(f'  - 超额年化收益: {strategy_metrics.annual_return - benchmark_metrics.annual_return:.2%}')
print(f'  - 信息比率: {(strategy_metrics.sharpe_ratio - benchmark_metrics.sharpe_ratio):.2f}')
print()
print('注意事项:')
print('  1. 本复现使用模拟数据，实际效果可能与研报存在差异')
  2. 部分原始研报中的数据指标在开源数据中不可得
  3. 历史回测结果不代表未来收益
  4. 真实策略需考虑交易成本、滑点、流动性等因素')
print()
print('=' * 70)

---

**参考研报**:
- 《行业配置策略：中观景气视角（1）》- 华泰证券 - 2022年1月
- 《行业配置策略：中观景气视角（2）》- 华泰证券 - 2022年7月